# Day 5-02｜成果圖表建立
> Python 籃球運動資料分析課程  
> 將角度資料、球軌跡與骨架示意整理成可展示的圖片。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 把分析表格轉成報告圖表。
- 建立角度折線圖與球軌跡圖。
- 輸出一張代表性的骨架示意圖。

## 完成產出
- 可放入成果展示的 PNG 圖片。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 載入或補齊分析資料。
2. 產生角度與軌跡圖。
3. 儲存展示圖片。


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

bootstrap_candidates = [
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
]
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，確認課程已同步到 /content/drive/MyDrive/basketball_hackathon/course。"
    )

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT, mount_drive=False)


In [ ]:
import pandas as pd
from src.shooting_utils import synthetic_pose_sequence, draw_skeleton
from src.plot_utils import plot_angle_series, plot_ball_path
from src.cv_utils import save_image_rgb, show_image

RESULTS = COURSE_ROOT / "assets" / "results"
pose_csv = RESULTS / "d4_02_pose_angles.csv"
ball_csv = RESULTS / "d4_03_ball_track.csv"

pose_df = pd.read_csv(pose_csv) if pose_csv.exists() else synthetic_pose_sequence(n=80)
ball_df = pd.read_csv(ball_csv) if ball_csv.exists() else pd.DataFrame()

In [ ]:
angle_png = RESULTS / "d5_02_pose_angles.png"
plot_angle_series(
    pose_df, ["elbow_angle", "knee_angle", "shoulder_angle"], output_path=angle_png
)
print("saved:", angle_png)

In [ ]:
if not ball_df.empty:
    ball_png = RESULTS / "d5_02_ball_path.png"
    plot_ball_path(ball_df, output_path=ball_png)
    print("saved:", ball_png)
else:
    print("沒有 ball_df，略過球軌跡圖。")

In [ ]:
# 選一個 elbow angle 最大的 frame 當展示骨架。
idx = (
    int(pose_df["elbow_angle"].idxmax())
    if "elbow_angle" in pose_df
    else len(pose_df) // 2
)
row = pose_df.iloc[idx]
skeleton = draw_skeleton(640, 480, row)
show_image(skeleton, f"showcase skeleton frame={int(row.frame)}")
skeleton_png = RESULTS / "d5_02_showcase_skeleton.png"
save_image_rgb(skeleton_png, skeleton)
print("saved:", skeleton_png)